In [1]:
from mssm.models import *
from mssmViz.sim import *
from mssmViz.plot import *
import pickle
import copy
import os
import time
from src.utils import init_lambda

size_conv = 2.54
single_width = 6/size_conv
double_width = 12/size_conv
full_width = 19/size_conv

should_plot = True # Whether plots should be generated to visually inspect simulation results

try:
    os.makedirs("./results")
except:
    print("Result directories already exist.")

try:
    os.makedirs("./results/sim")
    os.makedirs("./results/data")
    os.makedirs("./results/plots")
    os.makedirs("./results/data/sim1")
    os.makedirs("./results/data/sim2")
    os.makedirs("./results/data/sim3")
    os.makedirs("./results/data/sim1/plots")
    os.makedirs("./results/data/sim3/plots")
    os.makedirs("./results/data/sim4")
    os.makedirs("./results/data/sim5")
except:
    print("Result sub-directory already exist.")

Result directories already exist.
Result sub-directory already exist.


In [ ]:
############################# Simulation 1 #############################
n_sim = 100

sim_fams = [Gaussian(),Gamma(),Binomial()]
mod_fams = [Gaussian(),Gamma(),Binomial()]
mod2_fams = [GAUMLSS([Identity(),LOGb(-0.001)]),GAMMALS([LOG(),LOGb(-0.001)]),MULNOMLSS(1)]
fam_names = ["Gaussian", "Gamma", "Binom"]

for fam_name, mod_fam, mod_fam2, sim_fam in zip(fam_names,mod_fams,mod2_fams,sim_fams):
     
     # Set up storage for current sim
     eta_mses = np.zeros((n_sim,4))
     n_lam_updt = np.zeros((n_sim,4))
     Failures = np.zeros((n_sim,4))
     timings = np.zeros((n_sim,4))

     # Prevent gamm initialization for gammlss
     if fam_name != "Binom":
        mod_fam2.init_coef = lambda models: np.array([1e-4 for _ in range(models[0].formulas[0].n_coef + 1)]).reshape(-1,1)
     else:
        mod_fam2.init_coef = lambda models: np.array([1e-4 for _ in range(models[0].formulas[0].n_coef)]).reshape(-1,1)     

     mod_fam2.init_lambda = init_lambda

     gsmm_fam = GAMLSSGSMMFamily(2 if fam_name != "Binom" else 1,copy.deepcopy(mod_fam2))
     if fam_name != "Binom":
        gsmm_fam.init_coef = lambda models: np.array([1e-4 for _ in range(models[0].formulas[0].n_coef+1)]).reshape(-1,1)
     else:
        gsmm_fam.init_coef = lambda models: np.array([1e-4 for _ in range(models[0].formulas[0].n_coef)]).reshape(-1,1)

     gsmm_fam.init_lambda = init_lambda
     
     iterator = tqdm(range(n_sim),desc="Simulating",leave=True)
     for sim_i in iterator:

        sim_dat = sim11(5000,2,c=0,seed=sim_i,family=sim_fam,n_ranef=20,binom_offset = -5 if fam_name == "Binom" else 0)
        sim_dat.to_csv(f'./results/data/sim1/sim_size:{n_sim}_fam:{fam_name}_set:{sim_i}.csv',index=False)

        # We need to model the mean: \mu_i
        sim_formula_m = Formula(lhs("y"),
                            [i(),f(["x0"]),f(["x1"]),f(["x2"]),f(["x3"]),fs(["x0"],rf="x4")],
                            data=sim_dat)
        
        # And for sd - here constant
        sim_formula_sd = Formula(lhs("y"),
                            [i()],
                            data=sim_dat)
        
        if fam_name != "Binom":
            sim_formulas = [sim_formula_m,sim_formula_sd]
        else:
            sim_formulas = [sim_formula_m]

        ############################# Fit model with single inner iteration #############################
        sim_i_failed = [False, False, False, False]

        try:
            model_efs1 = GAMM(copy.deepcopy(sim_formula_m),mod_fam)
            start_time1 = time.process_time()
            model_efs1.fit(max_outer=1000,max_inner=1,method="Chol",progress_bar=False,n_cores=1)
            end_time1 = time.process_time()
            model_efs1.info.eps = 0
            
        except:
            sim_i_failed[0] = True
            model_efs1 = None

        ############################# Fit model with repeated inner iterattions #############################
        try:
            model_efs2 = GAMM(copy.deepcopy(sim_formula_m),copy.deepcopy(mod_fam))
            start_time2 = time.process_time()
            model_efs2.fit(max_outer=1000,max_inner=500,method="Chol",progress_bar=False,n_cores=1)
            end_time2 = time.process_time()
            model_efs2.info.eps = 0 
            
        except:
            sim_i_failed[1] = True
            model_efs2 = None

        ############################# Fit model with Newton #############################
        try:
            model_efs3 = GAMMLSS(copy.deepcopy(sim_formulas),copy.deepcopy(mod_fam2))
            
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                start_time3 = time.process_time()
                model_efs3.fit(seed=sim_i,max_outer=200,max_inner=500,min_inner=500,repara=True,method="LU/Chol",progress_bar=False,should_keep_drop=False,n_cores=1,prefit_grad=False)
                end_time3 = time.process_time()

        except:
            sim_i_failed[2] = True
            model_efs3 = None

        ############################# Fit model with qEFS #############################
        
        try:

            gsmm_model = GSMM(formulas=copy.deepcopy(sim_formulas),family=copy.deepcopy(gsmm_fam))

            # Fit with qEFS update without initialization
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                
                start_time_gsmm = time.process_time()
                gsmm_model.fit(init_coef=None,method='qEFS',extend_lambda=False,
                                control_lambda=1,max_outer=200,max_inner=500,min_inner=500,
                                seed=sim_i,qEFSH='SR1',overwrite_coef=False,qEFS_init_converge=False,prefit_grad=True,
                                progress_bar=False,repara=False,n_cores=1)
                end_time_gsmm = time.process_time()

            gsmm_model.info.eps = 0 # Not used but set to None, which messes up loop below
        except:
            sim_i_failed[3] = True
            gsmm_model = None
        
        ######################################## Collect MSEs ####################################
        models = [model_efs1,model_efs2,model_efs3,gsmm_model]
        start_times = [start_time1,start_time2,start_time3,start_time_gsmm]
        end_times = [end_time1,end_time2,end_time3,end_time_gsmm]

        if should_plot:
            fig = plt.figure(figsize=(full_width,1.25*single_width),layout='constrained')
            axs = fig.subplots(2,5,gridspec_kw=dict(wspace=0.05,hspace=0.01))

        for mi, model in enumerate(models):

            if sim_i_failed[mi]:
                print(f"Model {mi+1} failed at {sim_i}")
                Failures[sim_i,mi] = 1
                n_lam_updt[sim_i,mi] = np.nan
                eta_mses[sim_i,mi] = np.nan
                continue 

            if should_plot and mi > 1:
                plot(model,dist_par=0,axs=axs[mi-2])

            # Not converged but not failed outright
            if model.info.code > 0:
                Failures[sim_i,mi] = 1
            
            n_lam_updt[sim_i,mi] = model.info.iter
            timings[sim_i,mi] = end_times[mi] - start_times[mi]

            pred_diff = model.preds[0].flatten() - sim_dat["eta"].values
            eta_mses[sim_i,mi] = np.dot(pred_diff,pred_diff)/len(pred_diff)
        
        if should_plot:
            plt.savefig(f"./results/data/sim1/plots/sim_size:{n_sim}_fam:{fam_name}_set:{sim_i}.pdf", format="pdf", bbox_inches='tight')
            plt.close(fig)
        
        iterator.set_description_str(desc=f"MSE.: {[(float(np.round(m,decimals=4)),float(np.round(sd,decimals=2))) for m,sd in zip(np.mean(eta_mses[:(sim_i+1),:],axis=0),np.std(eta_mses[:(sim_i+1),:],axis=0))]}", refresh=True)
        ###################################### Save in progress results ######################################
        res = {"eta_mses":eta_mses,
               "n_lam_updt":n_lam_updt,
               "timings":timings,
               "Failures":Failures
               }
        
        with open(f'./results/sim/sim_1_size:{n_sim}_fam:{fam_name}.pickle', 'wb') as file:
            pickle.dump(res,file, protocol=pickle.HIGHEST_PROTOCOL)
            
     iterator.close()

MSE.: [(0.0968, 0.0), (0.0968, 0.0), (0.0968, 0.0), (0.1015, 0.0)]:   1%|          | 1/100 [01:13<2:01:13, 73.47s/it]

In [ ]:
############################# Simulation 1 #############################
n_sim = 100

# Set up storage for current sim
eta_mses = np.zeros((n_sim,2))
n_lam_updt = np.zeros((n_sim,2))
Failures = np.zeros((n_sim,2))
timings = np.zeros((n_sim,2))


iterator = tqdm(range(n_sim),desc="Simulating",leave=True)
for sim_i in iterator:

    sim_dat = sim11(5000,2,c=0,seed=sim_i,family=PropHaz([0],[0]),n_ranef=20,binom_offset = 0.1)

    # Prep everything for prophaz model
    sim_dat = sim_dat.sort_values(['y'],ascending=[False])
    sim_dat = sim_dat.reset_index(drop=True)

    u,inv = np.unique(sim_dat["y"],return_inverse=True)
    ut = np.flip(u)
    r = np.abs(inv - max(inv))
    
    sim_dat.to_csv(f'./results/data/sim1/sim_size:{n_sim}_fam:{"PropHaz"}_set:{sim_i}.csv',index=False)

    # We only need to model the mean: \mu_i
    sim_formula_m = Formula(lhs("delta"),
                        [f(["x0"]),f(["x1"]),f(["x2"]),f(["x3"]),fs(["x0"],rf="x4")],
                        data=sim_dat)

    sim_formulas = [sim_formula_m]

    sim_i_failed = [False, False]

    ############################# Fit model with Newton #############################
    try:
        gsmm_newton_fam = PropHaz(ut,r)
        gsmm_newton_fam.init_lambda = init_lambda
        gsmm_newton = GSMM(copy.deepcopy(sim_formulas),gsmm_newton_fam)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            start_time_newton = time.process_time()
            gsmm_newton.fit(init_coef=None,method="LU/Chol",extend_lambda=False,
                        control_lambda=2,max_outer=200,seed=sim_i,max_inner=500,
                        min_inner=500,progress_bar=False,prefit_grad=False,repara=True,n_cores=1)
            end_time_newton = time.process_time()

    except:
        sim_i_failed[0] = True
        gsmm_newton = None

    ############################# Fit model with qEFS #############################

    try:
        gsmm_qefs_fam = PropHaz(ut,r)
        gsmm_qefs_fam.init_lambda = init_lambda
        gsmm_qefs = GSMM(copy.deepcopy(sim_formulas),gsmm_qefs_fam)

        # Fit with qEFS update without initialization
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            
            start_time_qefs = time.process_time()
            gsmm_qefs.fit(init_coef=None,method='qEFS',extend_lambda=False,
                            control_lambda=1,max_outer=200,max_inner=500,min_inner=500,
                            seed=sim_i,qEFSH='SR1',overwrite_coef=False,qEFS_init_converge=False,prefit_grad=True,
                            progress_bar=False,repara=False,n_cores=1)
            end_time_qefs = time.process_time()


        gsmm_qefs.info.eps = 0 # Not used but set to None, which messes up loop below
    except:
        sim_i_failed[1] = True
        gsmm_qefs = None

    ######################################## Collect MSEs ####################################
    models = [gsmm_newton,gsmm_qefs]
    start_times = [start_time_newton,start_time_qefs]
    end_times = [end_time_newton,end_time_qefs]

    if should_plot:
        fig = plt.figure(figsize=(full_width,1.25*single_width),layout='constrained')
        axs = fig.subplots(2,5,gridspec_kw=dict(wspace=0.05,hspace=0.01))

    for mi, model in enumerate(models):

        if sim_i_failed[mi]:
            print(f"Model {mi+1} failed at {sim_i}")
            Failures[sim_i,mi] = 1
            n_lam_updt[sim_i,mi] = np.nan
            eta_mses[sim_i,mi] = np.nan
            continue 

        if should_plot:
            plot(model,dist_par=0,axs=axs[mi])

        # Not converged but not failed outright
        if model.info.code > 0:
            Failures[sim_i,mi] = 1
        
        n_lam_updt[sim_i,mi] = model.info.iter
        timings[sim_i,mi] = end_times[mi] - start_times[mi]

        pred_diff = model.preds[0].flatten() - sim_dat["eta"].values
        eta_mses[sim_i,mi] = np.dot(pred_diff,pred_diff)/len(pred_diff)
    
    if should_plot:
        plt.savefig(f'./results/data/sim1/plots/sim_size:{n_sim}_fam:{"PropHaz"}_set:{sim_i}.pdf', format="pdf", bbox_inches='tight')
        plt.close(fig)

    iterator.set_description_str(desc=f"MSE.: {[(float(np.round(m,decimals=4)),float(np.round(sd,decimals=2))) for m,sd in zip(np.mean(eta_mses[:(sim_i+1),:],axis=0),np.std(eta_mses[:(sim_i+1),:],axis=0))]}", refresh=True)
    ###################################### Save in progress results ######################################
    res = {"eta_mses":eta_mses,
            "n_lam_updt":n_lam_updt,
            "timings":timings,
            "Failures":Failures
            }

    with open(f'./results/sim/sim_1_size:{n_sim}_fam:{"PropHaz"}.pickle', 'wb') as file:
        pickle.dump(res,file, protocol=pickle.HIGHEST_PROTOCOL)

iterator.close()

Simulating:   0%|          | 0/100 [46:14<?, ?it/s]


NameError: name 'end_time_qefs' is not defined